# 04 · Contraste del PIP cacheado

## Objetivo
Repetir el PIP en pies de las cinco hojas con la misma función del expediente y publicar el valor redondeado a dos decimales.

## Alcance
La tolerancia de esta celda es relativa: la diferencia absoluta dividida por el PIP cacheado debe ser menor que 1e-9. No se ejecutan F-38 ni F-39.


In [1]:
from pathlib import Path
import os, sys
aqui = Path.cwd().resolve()
expediente = next(c for c in [aqui, *aqui.parents] if (c / "dataset.md").is_file() and (c / "entrada").is_dir())
os.chdir(expediente)
if str(expediente / "src") not in sys.path:
    sys.path.insert(0, str(expediente / "src"))
libro = expediente / "entrada" / "BES_jun.xlsx"

from ingesta.leer_bes_jun import leer_libro
from modelo_deterministico.fluidos_bes_jun import evaluar_fluido
from modelo_deterministico.balance_bes_jun import evaluar_balance
hojas = leer_libro(libro)
for h in hojas:
    fluido = evaluar_fluido(h["api_deg"], h["temperatura_c"], h["bsw_numero"], h["densidad_agua_lb_gal"])
    bal = evaluar_balance(
        h["presion_sensor_psi"], h["thp_psi"], h["tvd_intake_ft"], h["tvd_sensor_ft"],
        h["tvd_descarga_ft"], fluido.resultados["gradiente_psi_ft"]["valor"],
        h["caudal_operacion_bfpd"], h["id_tubing_in"],
    )
    pip = bal.resultados["pip_ft"]["valor"]
    diff_rel = abs(pip - h["cache_pip_ft"]) / h["cache_pip_ft"]
    print(h["pozo"], h["hoja"], "pip_ft", round(pip, 2), "diff_rel", diff_rel, "estado", bal.estado)
    assert bal.estado == "calculable"
    assert diff_rel < 1e-9


QF-084 QF-554H pip_ft 1714.5 diff_rel 0.0 estado calculable
QF-301 QF-545H pip_ft 1558.04 diff_rel 0.0 estado calculable
QF-147 QF-500H pip_ft 1920.73 diff_rel 0.0 estado calculable
QF-005 QF-378HST pip_ft 1891.21 diff_rel 0.0 estado calculable
QF-306 QF-549HST pip_ft 1422.32 diff_rel 0.0 estado calculable


## Resultado del contraste
Los PIP en pies, redondeados en la salida, son 1714.50 para QF-084, 1558.04 para QF-301, 1920.73 para QF-147, 1891.21 para QF-005 y 1422.32 para QF-306. La diferencia relativa impresa es 0 porque el valor calculado y el cache coinciden dentro de la precisión mostrada.

El documento `../../../docs/decision_modelo_base.md` sigue en borrador. Este contraste no lo aprueba. El PIP en pies sale de la presión de sensor; no es una predicción.
